# 03 — Pandas: EDA práctica y visualización con Matplotlib


Objetivos:

- Profundizar en **EDA** con pandas: valores nulos, duplicados, outliers.
- Joins y merges entre DataFrames.
- Visualización con **matplotlib** (histogramas, boxplots, barras, dispersión).


## 1. Cargar dataset sintético y provocar nulos/duplicados

**Qué es y para qué se usa.** Un `DataFrame` es una tabla etiquetada de pandas; sirve para reunir variables y observaciones y preparar una exploración reproducible. Aquí se construye un dataset sintético y se introducen problemas habituales para practicar su detección.

**Parámetros principales de las operaciones usadas.** En `np.random.default_rng(seed)`, `seed` fija la semilla; en `rng.integers(low, high, size)`, `low` y `high` delimitan el intervalo y `size` indica cuántos valores generar; en `rng.choice(a, size)` se eligen valores de `a`; y en `pd.concat(objs, ignore_index=True)`, `objs` son los objetos a concatenar e `ignore_index=True` renumera el índice.

**Ejemplo.** El bloque crea `df` con `pd.DataFrame({...})`, asigna `np.nan` a una muestra de `ingresos` y concatena cinco filas repetidas para disponer de un caso realista de EDA.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(7)
n = 300
df = pd.DataFrame({
    "id": range(1, n+1),
    "edad": rng.integers(18, 75, size=n),
    "ingresos": rng.normal(26000, 9000, size=n).round(2),
    "ciudad": rng.choice(["Cádiz","Sevilla","Málaga","Córdoba"], size=n),
    "satisfaccion": rng.integers(1, 6, size=n),
})
# Introducir nulos y duplicados
df.loc[rng.choice(df.index, 10, replace=False), "ingresos"] = np.nan
df = pd.concat([df, df.sample(5, random_state=1)], ignore_index=True)  # duplicados
df.head()

## 2. Nulos y duplicados

**Qué es y para qué se usa.** Los valores nulos representan información ausente y los duplicados, registros repetidos. Detectarlos antes del análisis evita estadísticas sesgadas y errores al combinar datos.

**Parámetros principales.** `isna()` identifica ausencias y `sum()` las cuenta por columna; `fillna(value)` sustituye nulos por `value` (aquí, la mediana); `drop_duplicates()` elimina filas repetidas y `reset_index(drop=True)` reconstruye el índice sin conservar el anterior.

**Ejemplo.** Primero se inspeccionan `df.isna().sum()` y `df.duplicated().sum()`; después se imputa `ingresos` y se eliminan duplicados. La última expresión comprueba que ambos problemas han quedado resueltos.

In [ ]:
df.isna().sum(), df.duplicated().sum()

In [ ]:
# Imputación simple
df["ingresos"] = df["ingresos"].fillna(df["ingresos"].median())
df = df.drop_duplicates().reset_index(drop=True)
df.isna().sum(), df.duplicated().sum()

## 3. Detección simple de outliers

**Qué es y para qué se usa.** Un *outlier* es una observación muy alejada del patrón central. Detectarlo ayuda a revisar errores de captura o fenómenos legítimos antes de decidir si se conserva, transforma o excluye.

**Parámetros principales.** `quantile(q)` calcula el cuantil `q`; el rango intercuartílico es `IQR = Q3 - Q1`. La regla habitual define límites `Q1 - 1.5·IQR` y `Q3 + 1.5·IQR`; la indexación booleana selecciona los registros fuera de esos límites.

**Ejemplo.** Se calculan `q1` y `q3` de `ingresos`, se construye la máscara con `|` (OR elemento a elemento) y se cuentan las filas seleccionadas en `outliers`.

In [ ]:
q1 = df["ingresos"].quantile(0.25)
q3 = df["ingresos"].quantile(0.75)
iqr = q3 - q1
lower = q1 - 1.5*iqr
upper = q3 + 1.5*iqr
outliers = df[(df["ingresos"] < lower) | (df["ingresos"] > upper)]
len(outliers), lower, upper

## 4. Joins / merges

**Qué es y para qué se usa.** Un *merge* relaciona filas de dos `DataFrame` mediante una clave común, como una JOIN de SQL. Permite enriquecer una tabla con atributos almacenados en otra.

**Parámetros principales.** En `df.merge(right, on, how)`, `right` es la tabla que se incorpora, `on` indica la columna clave compartida y `how` define la unión (`left` conserva todas las filas de la tabla izquierda; también existen `inner`, `right` y `outer`).

**Ejemplo.** `df.merge(ciudades_info, on="ciudad", how="left")` añade `poblacion_k` a cada persona y conserva todos los registros de `df`; si una clave no coincidiera, el nuevo valor sería nulo.

In [ ]:
ciudades_info = pd.DataFrame({
    "ciudad": ["Cádiz","Sevilla","Málaga","Córdoba"],
    "poblacion_k": [116, 684, 579, 326],
})
df_merged = df.merge(ciudades_info, on="ciudad", how="left")
df_merged.head()

## 5. Visualización con matplotlib

**Qué es y para qué se usa.** La visualización exploratoria representa distribuciones, comparaciones y relaciones para detectar patrones que no siempre aparecen en una tabla. pandas ofrece atajos que delegan el dibujo en Matplotlib.

**Parámetros principales de las llamadas usadas.** `plt.figure()` inicia una figura; `plot(kind="hist", bins, alpha)` controla el tipo, el número de intervalos y la transparencia; `df.boxplot(column, by, grid)` selecciona la variable, el grupo y la cuadrícula; `groupby("ciudad")` agrupa, `mean()` calcula la media y `sort_values()` ordena; `plot(kind="bar")` dibuja barras; `plt.scatter(x, y, alpha)` representa dos variables y su transparencia. `title`, `xlabel`, `ylabel` y `show` añaden contexto y muestran el gráfico.

**Ejemplo.** Los bloques comparan un histograma de `ingresos`, boxplots por `ciudad`, barras de satisfacción media y una dispersión de `edad` frente a `ingresos`; los títulos y etiquetas hacen explícita la lectura de cada figura.

In [ ]:
# Histograma de ingresos
plt.figure()
df["ingresos"].plot(kind="hist", bins=30, alpha=0.7)
plt.title("Histograma de ingresos")
plt.xlabel("Ingresos")
plt.ylabel("Frecuencia")
plt.show()

In [ ]:
# Boxplot por ciudad
plt.figure()
df.boxplot(column="ingresos", by="ciudad", grid=False)
plt.title("Ingresos por ciudad")
plt.suptitle("")
plt.xlabel("Ciudad")
plt.ylabel("Ingresos")
plt.show()

In [ ]:
# Barras: media de satisfacción por ciudad
media_sat = df.groupby("ciudad")["satisfaccion"].mean().sort_values()
plt.figure()
media_sat.plot(kind="bar")
plt.title("Satisfacción media por ciudad")
plt.xlabel("Ciudad")
plt.ylabel("Satisfacción media")
plt.show()

In [ ]:
# Dispersión edad vs ingresos
plt.figure()
plt.scatter(df["edad"], df["ingresos"], alpha=0.6)
plt.title("Edad vs Ingresos")
plt.xlabel("Edad")
plt.ylabel("Ingresos")
plt.show()

## 📝 Ejercicios

1) Repite el histograma de `edad` con 20 bins.  
2) Identifica outliers en `satisfaccion` usando IQR. ¿Hay muchos?  
3) Une (`merge`) con un DataFrame con código postal ficticio y comprueba el resultado.
